# 10 - Task 3: Rank-stacked ensemble of the linear and boosting families

Notebook 07 built an ensemble, measured no gain, and abandoned it. This one revisits that
conclusion, because 07's test could not have detected the effect even if it were there.

**07 tested the wrong members.** Its ensemble was ElasticNet + calibrated LinearSVC:
probability correlation **0.9745**, disagreeing on **11.4%** of dev rows. Notebook 08's
control B then confirmed ~0.001 gain at matched share. That is a result about *linear +
linear*. The one genuinely diverse member 07 tried, ComplementNB, was rejected for being
too weak at 0.6580 - so diversity and member quality were never available at once.

Notebook 09 supplied the missing pair. At matched share 0.4996, LightGBM disagrees with
ElasticNet on **956 test rows (13.7%)** while scoring **+0.0189** higher. That is the
first candidate pair which is simultaneously diverse *and* both-strong.

**07 also used the wrong instrument, twice.** Both errors follow from one fact: submissions
are made by `write_at_share`, an empirical quantile cut, so only the **ranking** of the
6,999 test rows matters.

1. *It averaged probabilities.* Rescaling one member monotonically is a no-op, but
   averaging raw probabilities is not rank-invariant - it weights each member by its
   probability spread, and LightGBM's probabilities are far sharper than a penalized
   logistic's. So 07's member weighting was an artifact of calibration. Here every member
   is converted to ranks first (`src/ensemble.py`).
2. *It selected on macro-F1 at a 0.5 cutoff* - the metric confounded by exactly the
   calibration effect 09 spent three rounds isolating. Here selection is on **OOF ROC AUC**,
   which *is* ranking quality and is invariant to the share.

Two consequences worth noting up front: LinearSVC no longer needs a `CalibratedClassifierCV`
wrapper (the expensive part of 07 - 5 inner fits per outer fold) because its raw margins
rank identically; and equal weights are not assumed, since the linear members are ~0.019
behind LightGBM and averaging them in at equal weight would very likely *hurt*.

**Honest expectation: +0.003 to +0.010.** The public-LB noise floor is ~0.0084 (09 section 6),
so this must be settled on the 16,000 OOF rows first, and only then cost a submission slot.

## 0. Setup

In [ ]:
# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json
import joblib

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from src import paths, data, evaluation, tuning, ensemble
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load features (sparse) + the locked split

In [ ]:
X, y, ids = data.load_train_features(sparse=True)
Xt, test_ids = data.load_test_features(sparse=True)
dev_idx = np.load(paths.DATA_PROCESSED / 'dev_idx.npy')
holdout_idx = np.load(paths.DATA_PROCESSED / 'holdout_idx.npy')
cv = evaluation.make_cv()

assert len(dev_idx) == 16000 and len(holdout_idx) == 4000
assert not set(dev_idx) & set(holdout_idx), "dev and holdout overlap"

X_dev, y_dev = X[dev_idx], y[dev_idx]
X_hold, y_hold = X[holdout_idx], y[holdout_idx]
print(f"dev {X_dev.shape}, holdout {X_hold.shape}, test {Xt.shape}")

## 2. The member registry, and who generates which

Every member is rebuilt from the `best_*_params.json` its own tuning notebook wrote, so
this notebook never re-searches and never drifts from what was tuned.

| member | source | why it is here |
|---|---|---|
| `lightgbm_v2` | `05f` winner, seed-averaged | strongest member |
| `xgboost` | `05e`, CV 0.7366 | already tuned and **never used anywhere** - free |
| `logreg_elasticnet` | `05c`, CV 0.7290 | the diverse linear direction |
| `linearsvc` | `05d`, CV 0.7275 | different objective (hinge, not log-loss); raw margins, no calibration |
| `extratrees` | new, untuned | never tried; decorrelated from boosting, and adds to Task 3's models-explored requirement |
| `complementnb` | untuned, CV 0.6580 | 07's most diverse member (22-28%) but weakest. Included so the **weight search can decide**, rather than being pre-rejected by an equal-weight vote that was always going to reject it |

**OOF generation is split three ways by cost.** Each of us runs only our own members and
pushes the resulting `.npy` files; section 4 merges whatever is on disk. **Change `ME` and
nothing else.**

In [ ]:
OOF_ASSIGNMENTS = {
    "cliffton": ["lightgbm_v2"],                                  # 5 seeds x 5 folds, the expensive one
    "jovyan":   ["xgboost", "extratrees"],                        # two moderate fits
    "brian":    ["logreg_elasticnet", "linearsvc", "complementnb"],  # saga is the slow one here
}

ME = "cliffton"  # <-- THE ONLY LINE TO CHANGE

SEEDS = [42, 43, 44, 45, 46]  # lightgbm_v2 seed-average
n_pos, n_neg = int((y_dev == 1).sum()), int((y_dev == 0).sum())


def load_params(name):
    with open(paths.DATA_PROCESSED / f"best_{name}_params.json") as f:
        p = json.load(f)
    # class_weight is fixed to "balanced" for every member below; drop it from the
    # loaded dict where a tuning notebook happened to persist it, so it is not passed twice.
    return {k: v for k, v in p.items() if k != "class_weight"}


def build(name, seed=42):
    '''Construct one member from its persisted tuned parameters.'''
    if name == "lightgbm_v2":
        return LGBMClassifier(class_weight="balanced", subsample_freq=1, verbose=-1,
                              n_jobs=-1, random_state=seed, **load_params("lightgbm_v2"))
    if name == "xgboost":
        return XGBClassifier(tree_method="hist", eval_metric="logloss",
                             scale_pos_weight=n_neg / n_pos, n_jobs=-1,
                             random_state=seed, **load_params("xgboost"))
    if name == "logreg_elasticnet":
        return LogisticRegression(penalty="elasticnet", solver="saga", max_iter=2000,
                                  class_weight="balanced", random_state=seed,
                                  **load_params("logreg_elasticnet"))
    if name == "linearsvc":
        # No CalibratedClassifierCV: the blend ranks members, and raw margins rank
        # identically to any monotone calibration of them.
        return LinearSVC(class_weight="balanced", max_iter=5000, dual="auto",
                         random_state=seed, **load_params("linearsvc"))
    if name == "extratrees":
        return ExtraTreesClassifier(n_estimators=500, class_weight="balanced",
                                    n_jobs=-1, random_state=seed)
    if name == "complementnb":
        return ComplementNB()
    raise KeyError(name)


ALL_MEMBERS = [m for members in OOF_ASSIGNMENTS.values() for m in members]
print(f"{ME} generates: {OOF_ASSIGNMENTS[ME]}")
print(f"full member set: {ALL_MEMBERS}")

## 3. Generate your out-of-fold scores

Out-of-fold predictions on `dev_idx` under the identical locked folds, so every member's
score and the blend's score are computed on the same rows and are directly comparable.

`lightgbm_v2` is additionally **seed-averaged** over 5 fits: the fits are highly correlated,
so rank-averaging them trims seed noise from the ranking without changing what the model is.
Near-free, typically +0.002-0.005, and it is an ensemble technique worth reporting in Task 4.

Results are written to `data/processed/oof/` and are **tracked in git** (64 KB per member),
so `git push` is what hands your work to the rest of the team.

In [ ]:
def oof_scores(est, X, y, cv):
    '''Out-of-fold machine-class score - probability, or margin for LinearSVC.'''
    method = "predict_proba" if hasattr(est, "predict_proba") else "decision_function"
    out = cross_val_predict(clone(est), X, y, cv=cv, method=method)
    return out[:, 1] if out.ndim == 2 else out


for name in OOF_ASSIGNMENTS[ME]:
    print(f"OOF: {name} ...", flush=True)
    if name == "lightgbm_v2":
        per_seed = [oof_scores(build(name, seed=s), X_dev, y_dev, cv) for s in SEEDS]
        scores = ensemble.seed_average(per_seed)
        spread = [evaluation.macro_f1(y_dev, (s >= 0.5).astype(int)) for s in per_seed]
        print(f"  per-seed F1: {np.round(spread, 4).tolist()}")
    else:
        scores = oof_scores(build(name), X_dev, y_dev, cv)

    path = tuning.save_oof(name, ME, scores)
    print(f"  AUC {roc_auc_score(y_dev, scores):.4f}  ->  {path.name}")

### Push your OOF files before continuing

```bash
git add data/processed/oof/
git commit -m "feat: OOF scores for <your members>"
git push
```

Sections 4 onward need **everyone's** files, so `git pull` first. Cliffton runs the rest.

## 4. Merge everyone's OOF scores

In [ ]:
print(tuning.available_oof().to_string(index=False))

missing = [m for m in ALL_MEMBERS if not list(paths.OOF.glob(f"{m}__*.npy"))]
assert not missing, f"still waiting on: {missing} - has everyone pushed?"

oof = {m: tuning.load_oof(m) for m in ALL_MEMBERS}
for m, s in oof.items():
    assert len(s) == len(dev_idx), (m, len(s))
    assert np.isfinite(s).all(), m
print(f"\n{len(oof)} members merged, {len(dev_idx)} rows each, no NaNs")

## 5. Member quality and diversity

`f1_at_share` thresholds each member so its predicted class balance equals the dev set's
true balance, rather than cutting at 0.5. That is the local mirror of `write_at_share` and
keeps this table free of the calibration confound - the numbers should sit close to each
member's tuned CV score.

The diversity matrices are the ones 07 reported, recomputed on the scale the combiner
actually uses: Spearman rank correlation, and disagreement at *matched* share. An ensemble
can only fix rows where members differ, so these bound the achievable gain.

In [ ]:
summary = ensemble.summarize_members(oof, y_dev, ALL_MEMBERS)
print(summary.to_string(index=False))

corr, disagree = ensemble.diversity_table(oof, y_dev, ALL_MEMBERS)
print("\nSpearman rank correlation:")
print(corr.round(4).to_string())
print("\nDisagreement at matched share:")
print(disagree.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, m, title in [(axes[0], corr, "Spearman rank correlation"),
                     (axes[1], disagree, "Disagreement at matched share")]:
    im = ax.imshow(m.to_numpy(), cmap="viridis")
    ax.set_xticks(range(len(m)), m.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(m)), m.index, fontsize=8)
    ax.set_title(title, fontsize=10)
    for i in range(len(m)):
        for j in range(len(m)):
            ax.text(j, i, f"{m.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if m.iloc[i, j] < m.to_numpy().mean() else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(FIGURES / "ensemble_member_diversity.png", dpi=120)
plt.show()

## 6. Weight search on OOF AUC

Two combiners, both on ranks:

- **Simplex sweep** - every weight vector on a 0.1 grid (3,003 of them for 6 members),
  scored by AUC. Exhaustive and interpretable; a member given weight 0 is one the ensemble
  does not want.
- **NNLS stacker** - `LinearRegression(positive=True)` on the OOF rank matrix. Non-negativity
  matters here because the members correlate 0.9+, and an unconstrained fit will assign
  large opposing weights that fit OOF noise and do not transport.

**The guard**, in 07's spirit: the blend has to beat the best single member by more than the
fold-to-fold AUC spread. If it does not, that is the finding, and the single model ships.

**One caveat to carry into the writeup.** Both combiners are *fitted on the same OOF rows
they are then scored on*, so the AUCs in this section are optimistically biased - the sweep
especially, since it takes the maximum over 3,003 weight vectors. The bias is small with
16,000 rows and 6 members, but it is not zero, and it is why section 7's holdout is the
unbiased read and why the guard is a spread rather than "any gain at all".

In [ ]:
R, members = ensemble.rank_matrix(oof, ALL_MEMBERS)

sweep = ensemble.search_weights(R, y_dev, step=0.1, metric=roc_auc_score)
w_sweep = sweep.iloc[0][[f"w{i}" for i in range(len(members))]].to_numpy(dtype=float)
w_stack = ensemble.fit_stacker(R, y_dev)

print("Top 5 weight vectors by OOF AUC:")
print(sweep.head(5).to_string(index=False))
print("\nweights            " + "  ".join(f"{m[:11]:>11s}" for m in members))
print("  sweep            " + "  ".join(f"{v:11.3f}" for v in w_sweep))
print("  nnls stacker     " + "  ".join(f"{v:11.3f}" for v in w_stack))

In [ ]:
def per_fold_auc(scores):
    '''AUC within each locked CV fold - the spread is this notebook's noise floor.'''
    dummy = np.zeros((len(y_dev), 1))
    return np.array([roc_auc_score(y_dev[te], scores[te])
                     for _, te in cv.split(dummy, y_dev)])


best_member = summary.iloc[0]["member"]
best_member_auc = float(summary.iloc[0]["auc"])
noise = float(per_fold_auc(oof[best_member]).std())

candidates = {"sweep": w_sweep, "nnls_stacker": w_stack,
              "equal_weight": np.full(len(members), 1 / len(members))}
rows = []
for label, w in candidates.items():
    blended = ensemble.blend(R, w)
    rows.append({"combiner": label,
                 "oof_auc": float(roc_auc_score(y_dev, blended)),
                 "oof_f1_at_share": ensemble.macro_f1_at_share(blended, y_dev, y_dev.mean())})
compare = pd.DataFrame(rows).sort_values("oof_auc", ascending=False).reset_index(drop=True)
compare["vs_best_member"] = compare["oof_auc"] - best_member_auc
print(compare.to_string(index=False))

chosen_label = compare.iloc[0]["combiner"]
chosen_w = candidates[chosen_label]
gain = float(compare.iloc[0]["vs_best_member"])
beats_best_member = gain > noise

print(f"\nBest single member: {best_member} (AUC {best_member_auc:.4f})")
print(f"Fold-to-fold AUC std (the bar): {noise:.4f}")
print(f"Chosen combiner: {chosen_label}, gain {gain:+.4f}")
print(f"Clears the bar: {beats_best_member}")
# Note for the writeup: 'equal_weight' is in the table specifically to show what 07's
# combiner would have scored on these members.

## 7. Holdout check

Members are fit on `dev_idx` only and evaluated once on the never-touched holdout.

Primary metric is **AUC**, which is share-free. The secondary F1 uses the holdout's *own*
0.6252 true share - the holdout is carved from train and inherits train's prior, not the
test set's ~0.53, so scoring it at any other share would reintroduce exactly the confound
notebook 09 removed.

In [ ]:
hold_scores = {}
for name in ALL_MEMBERS:
    if name == "lightgbm_v2":
        fits = [build(name, seed=s).fit(X_dev, y_dev) for s in SEEDS]
        hold_scores[name] = ensemble.seed_average(
            [ensemble.member_score(f, X_hold) for f in fits])
    else:
        hold_scores[name] = ensemble.member_score(build(name).fit(X_dev, y_dev), X_hold)
    print(f"{name:20s} holdout AUC {roc_auc_score(y_hold, hold_scores[name]):.4f}")

R_hold, _ = ensemble.rank_matrix(hold_scores, members)
blend_hold = ensemble.blend(R_hold, chosen_w)
hold_share = float(y_hold.mean())

print(f"\nEnsemble holdout AUC          {roc_auc_score(y_hold, blend_hold):.4f}")
print(f"Best member  holdout AUC      {roc_auc_score(y_hold, hold_scores[best_member]):.4f}")
print(f"Ensemble holdout F1 @ {hold_share:.4f} share  "
      f"{ensemble.macro_f1_at_share(blend_hold, y_hold, hold_share):.4f}")

## 8. Full refit and the share-matched submission batch

Benchmark: `lightgbm_share50.csv` at share 0.4996 scored **0.73583**. Every file below is
share-matched to *exactly* that, reusing notebook 09's control design, so any score
difference is purely model quality.

| # | file | isolates |
|---|---|---|
| 1 | `lightgbm_v2_share50.csv` | the `05f` retune, alone |
| 2 | `ensemble_share50.csv` | the ensemble, given #1 |
| 3 | `xgboost_share50.csv` | the never-submitted tuned XGBoost |

Uploading the blend without #1 would confound retuning and ensembling in one number - the
exact mistake 09 diagnosed. Hold the fourth slot; only once the model question resolves
should the share be re-placed (0.50 vs 0.53), and **share tuning itself stays closed** -
09 established that band is flat to within noise.

In [ ]:
BENCH_SHARE = 0.4996  # lightgbm_share50.csv, Kaggle 0.73583

sample = pd.read_csv(paths.DATA_RAW / "sample_submission.csv", dtype={"id": str})
assert len(test_ids) == 6999, len(test_ids)
assert list(test_ids) == list(sample["id"]), "test ids do not match sample_submission order"

test_scores = {}
fitted = {}
for name in ALL_MEMBERS:
    print(f"full refit: {name} ...", flush=True)
    if name == "lightgbm_v2":
        fits = [build(name, seed=s).fit(X, y) for s in SEEDS]
        fitted[name] = fits
        test_scores[name] = ensemble.seed_average(
            [ensemble.member_score(f, Xt) for f in fits])
    else:
        est = build(name).fit(X, y)
        fitted[name] = est
        test_scores[name] = ensemble.member_score(est, Xt)

joblib.dump(fitted, paths.MODELS / "round4_members_full_refit.pkl")
R_test, _ = ensemble.rank_matrix(test_scores, members)
blend_test = ensemble.blend(R_test, chosen_w)
print("\nrefit complete")

In [ ]:
def write_at_share(scores, target_share, filename):
    '''Threshold `scores` so exactly `target_share` of rows are predicted machine.'''
    thr = float(np.quantile(scores, 1 - target_share))
    preds = (scores >= thr).astype(int)
    data.write_submission(test_ids, preds, filename)
    return {"file": filename, "actual_share": round(float(preds.mean()), 4),
            "threshold": round(thr, 4)}


batch = [write_at_share(test_scores["lightgbm_v2"], BENCH_SHARE, "lightgbm_v2_share50.csv"),
         write_at_share(test_scores["xgboost"], BENCH_SHARE, "xgboost_share50.csv")]
if beats_best_member:
    batch.append(write_at_share(blend_test, BENCH_SHARE, "ensemble_share50.csv"))
else:
    print("Ensemble did NOT clear the fold-to-fold bar - no ensemble submission written.\n"
          "That is the finding; see the discussion section.\n")

batch = pd.DataFrame(batch)
bench = pd.read_csv(paths.SUBMISSIONS / "lightgbm_share50.csv")["label"].to_numpy()
batch["rows_vs_benchmark"] = [
    int((pd.read_csv(paths.SUBMISSIONS / f)["label"].to_numpy() != bench).sum())
    for f in batch["file"]
]
print(batch.to_string(index=False))
print("\nUnder ~200 differing rows the comparison is under-powered against the 0.0084 "
      "noise floor - a null result there must not be read as 'no difference' "
      "(09 section 9's share53 lesson).")

In [ ]:
# Format + correctness guards, matching 07/08/09. Run before uploading anything.
for f in batch["file"]:
    sub = pd.read_csv(paths.SUBMISSIONS / f, dtype={"id": str})
    assert list(sub.columns) == ["id", "label"], sub.columns
    assert list(sub["id"]) == list(sample["id"]), f"{f}: id order does not match sample"
    assert set(sub["label"].unique()) <= {0, 1}, f"{f}: labels outside 0/1"
    assert abs(sub["label"].mean() - BENCH_SHARE) < 1 / len(sub), f"{f}: share drifted"
    print(f"{f:28s} OK  n={len(sub)}  share={sub['label'].mean():.4f}")

# Identity check: a blend that puts all weight on one member must reproduce that member's
# own submission exactly. If this fails, the rank/weight plumbing is wrong.
solo = np.zeros(len(members)); solo[members.index("lightgbm_v2")] = 1.0
a = write_at_share(ensemble.blend(R_test, solo), BENCH_SHARE, "_identity_check.csv")
b = pd.read_csv(paths.SUBMISSIONS / "lightgbm_v2_share50.csv")["label"].to_numpy()
assert (pd.read_csv(paths.SUBMISSIONS / "_identity_check.csv")["label"].to_numpy() == b).all()
(paths.SUBMISSIONS / "_identity_check.csv").unlink()
print("\nidentity check passed: weight-1.0 blend == that member alone")

In [ ]:
ledger_path = paths.DATA_PROCESSED / "round4_results.csv"
ledger = batch[["file", "actual_share", "rows_vs_benchmark"]].copy()
ledger["kaggle_f1"] = np.nan
if ledger_path.exists():
    prev = pd.read_csv(ledger_path)[["file", "kaggle_f1"]].dropna()
    if len(prev):
        ledger = ledger.drop(columns="kaggle_f1").merge(prev, on="file", how="left")
ledger.to_csv(ledger_path, index=False)
print(ledger.to_string(index=False))
print(f"\nBenchmark to beat: lightgbm_share50.csv, share 0.4996, Kaggle 0.73583")
print(f"Ledger: {ledger_path}")

## Discussion / carry-forward -> Task 4 report

_Fill in once the batch is scored._

**What to record regardless of outcome.** Notebook 07 concluded the ensemble was not worth
having. This notebook re-ran that question with members chosen for measured diversity
(13.7% disagreement rather than 11.4% among near-identical linear models) and with two
instruments corrected - rank blending instead of probability averaging, AUC instead of
F1 at a fixed cutoff. The `equal_weight` row in section 6 shows directly what 07's combiner
would have scored on this member set, which makes the comparison concrete rather than
rhetorical.

**Read the batch as three separate questions:**

- `lightgbm_v2_share50` vs 0.73583 -> did re-searching LightGBM help? (05f alone.)
- `ensemble_share50` vs `lightgbm_v2_share50` -> did stacking help, on top of the retune?
- `xgboost_share50` vs 0.73583 -> a second boosting model at matched share, and the first
  score this project has ever obtained for its tuned XGBoost.

Anything inside +/-0.0084 is a tie, not a result. Say so plainly if that is what happened -
09 section 9 is the precedent, where a 0.0032 move was correctly called noise rather than
being written up as a regression.

**Still open, and a deliberate decision rather than an oversight:** the two final private-LB
picks. Both current picks are LightGBM near the same share, which is no hedge at all. Per
09 section 9, the right choice depends on what fraction of test rows the public leaderboard
scores - ~51% means a random split and both picks belong near the peak; 28.6% or 71.4% means
the split follows the UUID-vs-numeric `id` boundary, and pick 2 should hedge on *share* with
a train-matched 0.6252 variant instead.